# **KIỂM TRA GPU TRÊN COLAB**

In [ ]:
import sys
import platform
import subprocess

print("Hệ điều hành:", platform.platform())
print("Python:", sys.version)
print("\nThông tin GPU:")

result = subprocess.run(
    ["nvidia-smi"],
    capture_output=True,
    text=True
)
print(result.stdout or result.stderr)

In [ ]:
#Tạo môi trường Python 3.8

%%bash
set -euo pipefail

mkdir -p /content/hr17_tools

if [ ! -x /content/hr17_tools/bin/micromamba ]; then
    curl -fLsS \
      https://micro.mamba.pm/api/micromamba/linux-64/latest \
      -o /content/hr17_tools/micromamba.tar.bz2

    tar -xjf /content/hr17_tools/micromamba.tar.bz2 \
      -C /content/hr17_tools bin/micromamba
fi

export MAMBA_ROOT_PREFIX=/content/hr17_mamba

if [ ! -x /content/hr17_env/bin/python ]; then
    /content/hr17_tools/bin/micromamba create -y \
      -p /content/hr17_env \
      --override-channels -c conda-forge \
      python=3.8 pip
fi

/content/hr17_env/bin/python --version

# **TẠO MÔI TRƯỜNG VÀ KIỂM TRA PYTORCH NHẬN GPU**

In [ ]:
#Cài PyTorch theo phiên bản trong log cũ

%%bash
set -euo pipefail

/content/hr17_env/bin/python -m pip install \
  "pip==24.2" "numpy==1.23.0"

/content/hr17_env/bin/python -m pip install \
  torch==2.4.1 torchvision==0.19.1 \
  --index-url https://download.pytorch.org/whl/cu121

In [ ]:
#Kiểm tra tính toán trên CPU
%%bash
set -euo pipefail

/content/hr17_env/bin/python - <<'PY'
import sys
import numpy as np
import torch

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)
print("CUDA của PyTorch:", torch.version.cuda)
print("Nhận GPU:", torch.cuda.is_available())

assert torch.cuda.is_available(), "PyTorch chưa nhận GPU"

print("Tên GPU:", torch.cuda.get_device_name(0))

x = torch.ones((100, 100), device="cuda")
y = x @ x
torch.cuda.synchronize()

print("Kết quả kiểm tra:", y[0, 0].item())
print("ĐÃ KIỂM TRA PYTORCH TRÊN GPU THÀNH CÔNG")
PY

In [ ]:
#Kiểm tra và giải nén
from pathlib import Path
import tarfile

archive = Path("/content/IsaacGym_Preview_4_Package.tar")

assert archive.exists(), "Chưa thấy file. Hãy kiểm tra tải lên đã hoàn tất."

print("Dung lượng:", archive.stat().st_size, "bytes")
assert archive.stat().st_size == 201141673, (
    "Dung lượng khác file đã kiểm tra trên máy. Hãy kiểm tra lại."
)

with tarfile.open(archive, "r:*") as package:
    package.extractall("/content", filter="data")

assert Path("/content/isaacgym/python/setup.py").exists()
assert Path(
    "/content/isaacgym/python/isaacgym/_bindings/linux-x86_64/gym_38.so"
).exists()

print("ĐÃ GIẢI NÉN ISAAC GYM PREVIEW 4 THÀNH CÔNG")

In [ ]:
#Cài Isaac Gym vào Python 3.8:
%%bash
set -euo pipefail

/content/hr17_env/bin/python -m pip install \
  --no-build-isolation -e /content/isaacgym/python

export LD_LIBRARY_PATH="/content/hr17_env/lib:${LD_LIBRARY_PATH:-}"

/content/hr17_env/bin/python - <<'PY'
import isaacgym
from isaacgym import gymapi

gym = gymapi.acquire_gym()

print("Đường dẫn Isaac Gym:", isaacgym.__file__)
print("Acquire Gym:", gym is not None)
print("IMPORT ISAAC GYM THÀNH CÔNG")
PY

In [ ]:
#Kiểm tra simulator

%%bash
set -euo pipefail

export LD_LIBRARY_PATH="/content/hr17_env/lib:${LD_LIBRARY_PATH:-}"

 /content/hr17_env/bin/python -u - <<'PY' 2>&1 | tee /content/hr17_sim_check.log
from isaacgym import gymapi

gym = gymapi.acquire_gym()

# Cấu hình mô phỏng vật lý, không dùng cửa sổ đồ họa.
params = gymapi.SimParams()
params.dt = 1.0 / 60.0
params.substeps = 2
params.up_axis = gymapi.UP_AXIS_Z
params.gravity = gymapi.Vec3(0.0, 0.0, -9.81)
params.physx.use_gpu = True
params.use_gpu_pipeline = False

sim = gym.create_sim(0, -1, gymapi.SIM_PHYSX, params)
assert sim is not None, "Không tạo được simulator"

try:
    # Mặt đất.
    plane = gymapi.PlaneParams()
    plane.normal = gymapi.Vec3(0.0, 0.0, 1.0)
    gym.add_ground(sim, plane)

    # Một môi trường và khối hộp cạnh 0,2 m.
    env = gym.create_env(
        sim,
        gymapi.Vec3(-2.0, -2.0, 0.0),
        gymapi.Vec3(2.0, 2.0, 2.0),
        1,
    )

    options = gymapi.AssetOptions()
    options.fix_base_link = False
    box = gym.create_box(sim, 0.2, 0.2, 0.2, options)

    pose = gymapi.Transform()
    pose.p = gymapi.Vec3(0.0, 0.0, 1.0)

    actor = gym.create_actor(env, box, pose, "test_box", 0, 0)
    assert actor >= 0, "Không tạo được khối hộp"

    gym.prepare_sim(sim)

    print("Độ cao tâm hộp ban đầu: 1.000 m")

    for step in range(200):
        gym.simulate(sim)
        gym.fetch_results(sim, True)

        if step in (0, 29, 59, 199):
            states = gym.get_actor_rigid_body_states(
                env, actor, gymapi.STATE_POS
            )
            z = float(states["pose"]["p"]["z"][0])
            print(f"Bước {step + 1}: độ cao tâm hộp = {z:.3f} m")

    assert 0.03 < z < 0.20, (
        f"Hộp chưa nằm trên mặt đất như dự kiến: z={z}"
    )
    print("MÔ PHỎNG KHÔNG MỞ CỬA SỔ THÀNH CÔNG")

finally:
    gym.destroy_sim(sim)
PY

In [ ]:
#Tải repo và kiểm tra nạp robot h1

%%bash
set -euo pipefail

if [ ! -d /content/hr17_repo ]; then
    git clone https://github.com/LeCAR-Lab/human2humanoid.git \
      /content/hr17_repo
fi

git -C /content/hr17_repo checkout --detach 750f1fa

echo "Commit đang dùng:"
git -C /content/hr17_repo rev-parse --short HEAD

test -f /content/hr17_repo/legged_gym/resources/robots/h1/urdf/h1.urdf

echo "ĐÃ TÌM THẤY MÔ HÌNH H1"

In [ ]:
#Kiểm tra IssacGym đã nạp H1

%%bash
set -euo pipefail

export LD_LIBRARY_PATH="/content/hr17_env/lib:${LD_LIBRARY_PATH:-}"

/content/hr17_env/bin/python -u - <<'PY' 2>&1 | tee /content/hr17_h1_load_check.log
from pathlib import Path
from isaacgym import gymapi

asset_root = Path("/content/hr17_repo/legged_gym/resources")
asset_file = "robots/h1/urdf/h1.urdf"

assert (asset_root / asset_file).exists(), "Chưa tìm thấy URDF H1"

gym = gymapi.acquire_gym()

params = gymapi.SimParams()
params.dt = 1.0 / 60.0
params.up_axis = gymapi.UP_AXIS_Z
params.gravity = gymapi.Vec3(0.0, 0.0, -9.81)
params.physx.use_gpu = True
params.use_gpu_pipeline = False

sim = gym.create_sim(0, -1, gymapi.SIM_PHYSX, params)
assert sim is not None, "Không tạo được simulator"

try:
    options = gymapi.AssetOptions()
    options.fix_base_link = True

    asset = gym.load_asset(
        sim, str(asset_root), asset_file, options
    )
    assert asset is not None, "Không nạp được mô hình H1"

    dof_count = gym.get_asset_dof_count(asset)
    body_count = gym.get_asset_rigid_body_count(asset)
    dof_names = gym.get_asset_dof_names(asset)

    print("Số khớp điều khiển:", dof_count)
    print("Số rigid body:", body_count)
    print("Danh sách khớp:")
    for i, name in enumerate(dof_names):
        print(f"  {i}: {name}")

    assert dof_count == 19, (
        f"Mong đợi H1 có 19 DOF, nhưng nhận được {dof_count}"
    )

    env = gym.create_env(
        sim,
        gymapi.Vec3(-2.0, -2.0, 0.0),
        gymapi.Vec3(2.0, 2.0, 3.0),
        1,
    )

    pose = gymapi.Transform()
    pose.p = gymapi.Vec3(0.0, 0.0, 1.5)

    actor = gym.create_actor(env, asset, pose, "h1", 0, 0)
    assert actor >= 0, "Không tạo được robot H1 trong môi trường"

    gym.prepare_sim(sim)

    for _ in range(10):
        gym.simulate(sim)
        gym.fetch_results(sim, True)

    print("NẠP ROBOT H1 VÀ CHẠY MÔ PHỎNG THÀNH CÔNG")

finally:
    gym.destroy_sim(sim)
PY

In [ ]:
print("BỎ QUA: checkpoint cũ model_1000 không cần cho bước hiện tại.")

In [ ]:
#Kiểm tra mạng và thử tính action

%%bash
set -euo pipefail

/content/hr17_env/bin/python -u - <<'PY'
import json
from pathlib import Path
import torch
from torch import nn

folder = Path("/content/hr17_checkpoint")

env_cfg = json.loads((folder / "env_cfg.json").read_text())
train_cfg = json.loads((folder / "train_cfg.json").read_text())

num_obs = env_cfg["env"]["num_observations"]
num_actions = env_cfg["env"]["num_actions"]
hidden_dims = train_cfg["policy"]["actor_hidden_dims"]

print("Số chiều quan sát:", num_obs)
print("Số action:", num_actions)
print("Các lớp ẩn:", hidden_dims)

assert num_obs == 138
assert num_actions == 19
assert train_cfg["policy"]["activation"].lower() == "elu"

# Chỉ dùng cơ chế đọc trọng số hạn chế của PyTorch.
checkpoint = torch.load(
    folder / "model_1000.pt",
    map_location="cpu",
    weights_only=True,
)

state = checkpoint["model_state_dict"]
actor_state = {
    key[len("actor."):]: value
    for key, value in state.items()
    if key.startswith("actor.")
}
assert actor_state, "Không tìm thấy trọng số actor"

dims = [num_obs] + hidden_dims + [num_actions]
layers = []

for i in range(len(dims) - 1):
    layers.append(nn.Linear(dims[i], dims[i + 1]))
    if i < len(dims) - 2:
        layers.append(nn.ELU())

actor = nn.Sequential(*layers)
actor.load_state_dict(actor_state, strict=True)
actor.eval()

with torch.inference_mode():
    test_obs = torch.zeros(1, num_obs)
    actions = actor(test_obs)

assert actions.shape == (1, 19)
assert torch.isfinite(actions).all()

print("Kích thước action:", tuple(actions.shape))
print("Action đều hữu hạn:", torch.isfinite(actions).all().item())
print("CHECKPOINT KHỚP MẠNG VÀ TÍNH ĐƯỢC ACTION")
PY

**↪ Đọc trọng số và thử một đầu vào giả; không huấn luyện hoặc điều khiển robot.**

**Checkpoint đã qua kiểm tra cấu trúc:** mạng [138 → 512 → 256 → 128 → 19] load được trọng số và tính được action hữu hạn. Chưa thể kết luận chất lượng điều khiển từ đầu vào giả này.

In [ ]:
#Cài nhóm thư viện cơ bản của repo

%%bash
set -euo pipefail

# Cố định các phiên bản đang hoạt động.
cat > /content/hr17_constraints.txt <<'EOF'
torch==2.4.1+cu121
torchvision==0.19.1+cu121
numpy==1.23.0
scipy==1.10.1
EOF

/content/hr17_env/bin/python -m pip install \
  -c /content/hr17_constraints.txt \
  scipy==1.10.1 \
  matplotlib==3.7.5 \
  joblib==1.4.2 \
  hydra-core==1.3.2 \
  easydict==1.13 \
  tensorboard==2.14.0 \
  wandb==0.17.9 \
  tqdm pyyaml termcolor rich ipdb lxml

# Đăng ký các package trong repo.
# Dependencies được cài theo từng nhóm để kiểm soát phiên bản.
cd /content/hr17_repo

/content/hr17_env/bin/python -m pip install \
  --no-deps --no-build-isolation \
  -e rsl_rl -e legged_gym -e phc

echo "ĐÃ CÀI NHÓM THƯ VIỆN CƠ BẢN"

In [ ]:
%%bash
set -euo pipefail

export LD_LIBRARY_PATH="/content/hr17_env/lib:${LD_LIBRARY_PATH:-}"

/content/hr17_env/bin/python - <<'PY'
import isaacgym
import torch
import numpy as np
import scipy
import hydra
import joblib
import matplotlib
from easydict import EasyDict
from torch.utils.tensorboard import SummaryWriter
from importlib.metadata import version

print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("Nhận GPU:", torch.cuda.is_available())

for name in ("rsl_rl", "legged_gym", "phc"):
    print(f"Package {name}:", version(name))

assert torch.__version__ == "2.4.1+cu121"
assert np.__version__ == "1.23.0"
assert torch.cuda.is_available()

print("NHÓM THƯ VIỆN CƠ BẢN ĐÃ SẴN SÀNG")
PY

In [ ]:
#Cài các phụ thuộc bổ sung
%%bash
set -euo pipefail

/content/hr17_env/bin/python -m pip install \
  -c /content/hr17_constraints.txt \
  gym==0.23.1 \
  rl-games==1.6.1 \
  ninja==1.11.1.1

mkdir -p /content/hr17_deps

if [ ! -d /content/hr17_deps/SMPLSim ]; then
    git clone --depth 1 https://github.com/ZhengyiLuo/SMPLSim.git \
      /content/hr17_deps/SMPLSim
fi

if [ ! -d /content/hr17_deps/smplx ]; then
    git clone --depth 1 https://github.com/ZhengyiLuo/smplx.git \
      /content/hr17_deps/smplx
fi

/content/hr17_env/bin/python -m pip install \
  --no-deps --no-build-isolation \
  -e /content/hr17_deps/SMPLSim \
  -e /content/hr17_deps/smplx

{
    echo "SMPLSim:"
    git -C /content/hr17_deps/SMPLSim rev-parse HEAD
    echo "smplx:"
    git -C /content/hr17_deps/smplx rev-parse HEAD
} | tee /content/hr17_dependency_commits.txt

echo "Ô 1 HOÀN TẤT"

In [ ]:
#Điều chỉnh hai import và lưu bản thay đổi

%%bash
set -euo pipefail

/content/hr17_env/bin/python - <<'PY'
from pathlib import Path

# 1. Chuyển import môi trường MuJoCo sang lúc gọi hàm make().
path = Path("/content/hr17_deps/SMPLSim/smpl_sim/__init__.py")
text = path.read_text()
old = "from . import envs as domains"

if text.startswith(old):
    anchor = "def make(cfg, motions):"
    assert text.count(anchor) == 1, "Cấu trúc SMPLSim khác dự kiến"

    backup = Path(str(path) + ".hr17_original")
    if not backup.exists():
        backup.write_text(text)

    text = text.replace(old, "# HR17: import envs inside make()", 1)
    text = text.replace(
        anchor,
        anchor + "\n    from . import envs as domains",
        1,
    )
    path.write_text(text)
else:
    assert "# HR17: import envs inside make()" in text

# 2. Dùng các hàm rotation có sẵn trong PHC.
path = Path("/content/hr17_repo/legged_gym/legged_gym/utils/transform.py")
text = path.read_text()
old = "from pytorch3d.transforms import"
new = "from phc.utils.rotation_conversions import"

if old in text:
    backup = Path(str(path) + ".hr17_original")
    if not backup.exists():
        backup.write_text(text)
    path.write_text(text.replace(old, new, 1))
else:
    assert new in text

print("ĐÃ ĐIỀU CHỈNH IMPORT VÀ LƯU BẢN GỐC")
PY

git -C /content/hr17_repo diff \
  > /content/hr17_repo_changes.patch

git -C /content/hr17_deps/SMPLSim diff \
  > /content/hr17_smplsim_changes.patch

In [ ]:
#Kiểm tra môi trường H2O

%%bash
set -euo pipefail

# Cho phép tìm ninja và các chương trình trong môi trường Python 3.8.
export PATH="/content/hr17_env/bin:$PATH"
export LD_LIBRARY_PATH="/content/hr17_env/lib:${LD_LIBRARY_PATH:-}"
export MAX_JOBS=2
export MPLBACKEND=Agg

echo "Đường dẫn Ninja:"
command -v ninja

echo "Phiên bản Ninja:"
ninja --version

cd /content/hr17_repo

/content/hr17_env/bin/python -u - <<'PY' 2>&1 | tee /content/hr17_import_check.log
import isaacgym
import torch
import numpy as np

from smplx import SMPL, SMPLH, SMPLX
from smpl_sim.poselib.skeleton.skeleton3d import SkeletonTree
print("SMPL-X và SkeletonTree: OK")

from phc.utils.motion_lib_h1 import MotionLibH1
print("MotionLibH1: OK")

from legged_gym.envs import LeggedRobot
from legged_gym.utils import task_registry
print("LeggedRobot:", LeggedRobot.__name__)

assert torch.cuda.is_available()
assert np.__version__ == "1.23.0"

print("IMPORT MÔI TRƯỜNG H2O THÀNH CÔNG")
PY

In [ ]:
#Chạy thử checkpoint trong một môi trường với stable_punch.pkl

%%bash
set -euo pipefail

export PATH="/content/hr17_env/bin:$PATH"
export LD_LIBRARY_PATH="/content/hr17_env/lib:${LD_LIBRARY_PATH:-}"
export MAX_JOBS=2
export MPLBACKEND=Agg

cd /content/hr17_repo

/content/hr17_env/bin/python -u - <<'PY' 2>&1 | tee /content/hr17_checkpoint_smoke.log
import json
from pathlib import Path

import isaacgym
from isaacgym import gymapi, gymutil
import torch
from torch import nn
from easydict import EasyDict

from legged_gym.envs import LeggedRobot
from legged_gym.utils.helpers import set_seed

repo = Path("/content/hr17_repo")
folder = Path("/content/hr17_checkpoint")
output = Path("/content/hr17_smoke")
output.mkdir(exist_ok=True)

# Đọc lại cấu hình gốc.
cfg = EasyDict(json.loads((folder / "env_cfg.json").read_text()))
train_cfg = json.loads((folder / "train_cfg.json").read_text())

# Chạy một môi trường, không mở cửa sổ.
cfg.num_envs = 1
cfg.env.num_envs = 1
cfg.headless = True
cfg.sim_device = "cuda:0"
cfg.rl_device = "cuda:0"
cfg.test = True
cfg.env.test = True
cfg.env.im_eval = False
cfg.train.distill = False
cfg.use_wandb = False
cfg.viewer.debug_viz = False
cfg.motion.visualize = False
cfg.motion.realtime_vr_keypoints = False

# Đường dẫn tới mô hình và motion.
resources = repo / "legged_gym" / "resources"
cfg.asset.file = str(resources / "robots/h1/urdf/h1.urdf")
cfg.motion.motion_file = str(resources / "motions/h1/stable_punch.pkl")
cfg.motion.skeleton_file = str(resources / "robots/h1/xml/h1.xml")
cfg.motion.marker_file = str(
    resources / "objects/Marker/traj_marker.urdf"
)

assert Path(cfg.asset.file).exists(), f"Thiếu file: {cfg.asset.file}"
for name in ("motion_file", "skeleton_file"):
    assert Path(cfg.motion[name]).exists(), (
        f"Thiếu file: {cfg.motion[name]}"
    )

# Địa hình phẳng dạng trimesh nhỏ.
cfg.terrain.curriculum = False
cfg.terrain.mesh_type = "trimesh"
cfg.terrain.num_rows = 1
cfg.terrain.num_cols = 1
cfg.terrain.max_init_terrain_level = 0
cfg.terrain.border_size = 5
cfg.terrain.terrain_types = ["flat"]
cfg.terrain.terrain_proportions = [1.0]

# Chỉ tắt các cờ ngẫu nhiên kiểu bool.
cfg.noise.add_noise = False
for key, value in list(cfg.domain_rand.items()):
    if key.startswith("randomize_") and isinstance(value, bool):
        cfg.domain_rand[key] = False

assert isinstance(
    cfg.domain_rand.randomize_link_body_names, list
), "Danh sách khâu robot phải được giữ nguyên"

cfg.domain_rand.push_robots = False
cfg.domain_rand.motion_package_loss = False
cfg.domain_rand.born_offset = False
cfg.domain_rand.born_heading_randomization = False
cfg.motion.resample_motions_for_envs = False

(output / "config_requested.json").write_text(
    json.dumps(cfg, indent=2)
)

set_seed(cfg.seed)

sim_params = gymapi.SimParams()
gymutil.parse_sim_config(cfg.sim, sim_params)
sim_params.use_gpu_pipeline = True
sim_params.physx.use_gpu = True

# Khôi phục actor, không nạp optimizer.
checkpoint = torch.load(
    folder / "model_1000.pt",
    map_location="cpu",
    weights_only=True,
)

dims = (
    [cfg.env.num_observations]
    + train_cfg["policy"]["actor_hidden_dims"]
    + [cfg.env.num_actions]
)
assert train_cfg["policy"]["activation"].lower() == "elu"

layers = []
for i in range(len(dims) - 1):
    layers.append(nn.Linear(dims[i], dims[i + 1]))
    if i < len(dims) - 2:
        layers.append(nn.ELU())

actor = nn.Sequential(*layers)
actor.load_state_dict({
    key[len("actor."):]: value
    for key, value in checkpoint["model_state_dict"].items()
    if key.startswith("actor.")
}, strict=True)
actor = actor.to("cuda:0").eval()

print("ĐANG KHỞI TẠO MÔI TRƯỜNG H2O...")
env = LeggedRobot(
    cfg=cfg,
    sim_params=sim_params,
    physics_engine=gymapi.SIM_PHYSX,
    sim_device="cuda:0",
    headless=True,
)

try:
    obs = env.get_observations()
    print("Kích thước quan sát:", tuple(obs.shape))
    print("Chu kỳ điều khiển:", env.dt, "giây")

    assert obs.shape == (1, 138)
    resets = 0

    with torch.inference_mode():
        for step in range(100):
            assert torch.isfinite(obs).all(), "Quan sát có NaN/Inf"

            actions = actor(obs)
            assert torch.isfinite(actions).all(), "Action có NaN/Inf"

            obs, _, rewards, dones, infos = env.step(actions)
            assert torch.isfinite(rewards).all(), "Reward có NaN/Inf"

            resets += int(dones.sum().item())

            if (step + 1) % 25 == 0:
                print(
                    f"Bước {step + 1}/100 | "
                    f"reward={rewards.mean().item():.4f} | "
                    f"số lần reset tích lũy={resets}"
                )

    summary = {
        "completed_steps": 100,
        "reset_events": resets,
        "checkpoint": "model_1000.pt",
        "purpose": "technical smoke test, not formal evaluation",
    }
    (output / "summary.json").write_text(
        json.dumps(summary, indent=2)
    )

    print("CHẠY THỬ CHECKPOINT H2O THÀNH CÔNG")

finally:
    env.gym.destroy_sim(env.sim)
PY

In [ ]:
from pathlib import Path
import subprocess

repo = Path("/content/hr17_repo")
path = repo / "legged_gym/legged_gym/envs/base/legged_robot.py"
text = path.read_text()

old = (
    "self.projected_gravity[:] = "
    "quat_rotate_inverse(self._rigid_body_rot[:, 11, :], self.clear)"
)
new = (
    "self.projected_gravity[:] = "
    "quat_rotate_inverse(self._rigid_body_rot[:, 11, :], self.gravity_vec)"
)

if old in text:
    assert text.count(old) == 1, "Số dòng khớp khác dự kiến"

    backup = path.with_name("legged_robot.py.hr17_before_gravity_fix")
    if not backup.exists():
        backup.write_text(text)

    updated = text.replace(old, new, 1)
    compile(updated, str(path), "exec")
    path.write_text(updated)
    print("ĐÃ SỬA: self.clear → self.gravity_vec")

else:
    assert new in text, "Không tìm thấy dòng cần sửa. Hãy gửi output."
    print("Dòng này đã được sửa trước đó.")

result = subprocess.run(
    ["git", "-C", str(repo), "diff"],
    check=True,
    capture_output=True,
    text=True,
)
Path("/content/hr17_repo_changes.patch").write_text(result.stdout)

print("Đã lưu thay đổi vào /content/hr17_repo_changes.patch")

In [ ]:
from pathlib import Path
import zipfile
from google.colab import files

paths = [
    Path("/content/hr17_checkpoint_smoke.log"),
    Path("/content/hr17_smoke/config_requested.json"),
    Path("/content/hr17_smoke/summary.json"),
    Path("/content/hr17_repo_changes.patch"),
    Path("/content/hr17_smplsim_changes.patch"),
    Path("/content/hr17_dependency_commits.txt"),
]

missing = [str(p) for p in paths if not p.exists()]
if missing:
    print("Các file chưa tìm thấy:")
    for name in missing:
        print("-", name)
else:
    archive = Path("/content/HR17_ket_qua_chay_thu.zip")

    with zipfile.ZipFile(
        archive, "w", compression=zipfile.ZIP_DEFLATED
    ) as package:
        for path in paths:
            package.write(path, arcname=path.relative_to("/content"))

    print("Đã đóng gói đầy đủ:", archive.name)
    files.download(str(archive))

In [ ]:
#Tìm hiểu nguyên nhân 10 lần reset

%%bash
set -euo pipefail

export PATH="/content/hr17_env/bin:$PATH"
export LD_LIBRARY_PATH="/content/hr17_env/lib:${LD_LIBRARY_PATH:-}"
export MAX_JOBS=2
export MPLBACKEND=Agg

if [ ! -x /content/hr17_env/bin/python ]; then
    echo "Phiên Colab hiện tại không còn môi trường Python đã cài."
    exit 1
fi

HR17_DIAG_SCRIPT="/content/hr17_diagnose.py"

if [ ! -f "$HR17_DIAG_SCRIPT" ]; then
    HR17_DIAG_SCRIPT="/hr17_diagnose.py"
fi

if [ ! -f "$HR17_DIAG_SCRIPT" ]; then
    echo "Chưa tìm thấy hr17_diagnose.py. Hãy kiểm tra tải lên."
    exit 1
fi

cd /content/hr17_repo

/content/hr17_env/bin/python -u "$HR17_DIAG_SCRIPT" \
  2>&1 | tee /content/hr17_diagnosis_console.log

Điều đó nghĩa là thành phần X của vector trọng lực trong hệ tọa độ thân robot vượt ngưỡng cho phép. **Chưa đủ để kết luận** robot đã ngã xuống đất; cần xem trạng thái trước reset để biết vì sao robot nghiêng nhanh như vậy.

| Nội dung | Kết quả |
|---|---:|
| Thời điểm reset mỗi episode | Bước 10, khoảng **0,2 giây** |
| Thời lượng motion tham chiếu | **5,233 giây** |
| Nguyên nhân reset | `gravity_x`: 10 lần |
| MPJPE toàn cục trên các frame đã ghi | **101,49 mm** |
| MPJPE tương đối với gốc | **97,14 mm** |
| Success rate | Chưa đo theo quy trình đầy đủ |

In [ ]:
#Đọc dữ liệu từng Frame để kiểm tra
from pathlib import Path
import zipfile
from google.colab import files

folder = Path("/content/hr17_diagnosis")
archive = Path("/content/HR17_chan_doan_reset.zip")

assert (folder / "frames.csv").exists(), "Chưa tìm thấy frames.csv"
assert (folder / "summary.json").exists(), "Chưa tìm thấy summary.json"

with zipfile.ZipFile(
    archive, "w", compression=zipfile.ZIP_DEFLATED
) as package:
    for path in folder.rglob("*"):
        if path.is_file():
            package.write(
                path,
                arcname=Path("hr17_diagnosis") / path.relative_to(folder),
            )

    log = Path("/content/hr17_diagnosis_console.log")
    if log.exists():
        package.write(log, arcname=log.name)

files.download(str(archive))

In [ ]:
%%bash
set -euo pipefail

export PATH="/content/hr17_env/bin:$PATH"
export LD_LIBRARY_PATH="/content/hr17_env/lib:${LD_LIBRARY_PATH:-}"
export MAX_JOBS=2
export MPLBACKEND=Agg

HR17_SCRIPT="/content/hr17_compare_controls.py"
if [ ! -f "$HR17_SCRIPT" ]; then
    HR17_SCRIPT="/hr17_compare_controls.py"
fi

test -f "$HR17_SCRIPT"
test -x /content/hr17_env/bin/python

mkdir -p /content/hr17_comparison
cd /content/hr17_repo

for mode in checkpoint zero reference_pd; do
    /content/hr17_env/bin/python -u "$HR17_SCRIPT" \
      --controller "$mode" \
      2>&1 | tee "/content/hr17_comparison/${mode}_console.log"
done

/content/hr17_env/bin/python - <<'PY'
import json
from pathlib import Path

print("\nKẾT QUẢ SO SÁNH:")
for mode in ("checkpoint", "zero", "reference_pd"):
    path = Path("/content/hr17_comparison") / mode / "summary.json"
    data = json.loads(path.read_text())
    print(
        mode,
        "| reset:", data["reset_count"],
        "| nguyên nhân:", data["reset_reason_counts"],
        "| MPJPE các frame đã ghi:",
        round(data["observed_frame_global_mpjpe_mm"], 2), "mm",
    )
PY

↪Kết quả cho thấy checkpoint hiện tại **mất ổn định** sớm hơn hai cách điều khiển đối chứng. Tuy nhiên, cả ba cách đều có reset, nên **chưa thể kết luận** nguyên nhân chỉ nằm ở checkpoint.

In [ ]:
#Phân tích dữ liệu đã có trên cùng 10 bước đầu

from pathlib import Path
import csv
import json
import zipfile
from statistics import mean
from google.colab import files

folder = Path("/content/hr17_comparison")
modes = ("checkpoint", "zero", "reference_pd")
traces = {}

for mode in modes:
    path = folder / mode / "frames.csv"
    assert path.exists(), f"Thiếu file: {path}"

    with path.open() as stream:
        rows = list(csv.DictReader(stream))

    first_episode = [r for r in rows if int(r["episode"]) == 0]
    assert first_episode, f"Không có dữ liệu episode đầu: {mode}"
    traces[mode] = first_episode

# So sánh trên cùng khoảng thời gian trước lần reset sớm nhất.
n = min(len(rows) for rows in traces.values())
comparison = {}

print(f"SO SÁNH CÙNG {n} BƯỚC ĐẦU:")
for mode, rows in traces.items():
    selected = rows[:n]
    result = {
        "steps": n,
        "global_mpjpe_mm": mean(
            float(r["global_mpjpe_mm"]) for r in selected
        ),
        "root_relative_mpjpe_mm": mean(
            float(r["root_relative_mpjpe_mm"]) for r in selected
        ),
        "last_root_height_m": float(selected[-1]["root_height_m"]),
        "last_gravity_x": float(selected[-1]["gravity_x"]),
    }
    comparison[mode] = result
    print(mode, json.dumps(result, ensure_ascii=False))

(folder / "common_horizon_summary.json").write_text(
    json.dumps(comparison, indent=2)
)

archive = Path("/content/HR17_so_sanh_dieu_khien.zip")
with zipfile.ZipFile(
    archive, "w", compression=zipfile.ZIP_DEFLATED
) as package:
    for path in folder.rglob("*"):
        if path.is_file():
            package.write(path, arcname=path.relative_to("/content"))

files.download(str(archive))

In [ ]:
#Chạy giải nén

from pathlib import Path
import zipfile
import json
import hashlib

name = "HR17_so_sanh_checkpoint_500_1000.zip"
archive = next(
    (p for p in (Path("/content") / name, Path("/") / name)
     if p.exists()),
    None,
)
assert archive is not None, "Chưa tìm thấy ZIP. Hãy kiểm tra tải lên."

output = Path("/content/hr17_checkpoint_comparison_input")
output.mkdir(exist_ok=True)

expected = (
    "model_500.pt",
    "model_1000.pt",
    "env_cfg.json",
    "train_cfg.json",
    "hr17_compare_checkpoints.py",
)

with zipfile.ZipFile(archive) as package:
    manifest = json.loads(package.read("manifest.json"))

    for filename in expected:
        data = package.read(filename)
        assert hashlib.sha256(data).hexdigest() == (
            manifest[filename]["sha256"]
        ), f"File không khớp: {filename}"

        (output / filename).write_bytes(data)
        print("Đã kiểm tra:", filename)

print("ĐÃ CHUẨN BỊ HAI CHECKPOINT")



In [ ]:
#So sánh model_500.pt với model_1000.pt
%%bash
set -euo pipefail

export PATH="/content/hr17_env/bin:$PATH"
export LD_LIBRARY_PATH="/content/hr17_env/lib:${LD_LIBRARY_PATH:-}"
export MAX_JOBS=2
export MPLBACKEND=Agg

test -x /content/hr17_env/bin/python

mkdir -p /content/hr17_checkpoint_comparison
cd /content/hr17_repo

for number in 500 1000; do
    /content/hr17_env/bin/python -u \
      /content/hr17_checkpoint_comparison_input/hr17_compare_checkpoints.py \
      --checkpoint "model_${number}.pt" \
      2>&1 | tee \
      "/content/hr17_checkpoint_comparison/model_${number}_console.log"
done

/content/hr17_env/bin/python - <<'PY'
import csv
import json
from pathlib import Path
from statistics import mean

folder = Path("/content/hr17_checkpoint_comparison")
traces = {}
summaries = {}

for name in ("model_500", "model_1000"):
    summaries[name] = json.loads(
        (folder / name / "summary.json").read_text()
    )

    with (folder / name / "frames.csv").open() as stream:
        rows = list(csv.DictReader(stream))

    traces[name] = [r for r in rows if int(r["episode"]) == 0]

n = min(len(rows) for rows in traces.values())
assert n > 0

results = {}
print(f"\nKẾT QUẢ SO SÁNH TRÊN CÙNG {n} BƯỚC ĐẦU:")

for name, rows in traces.items():
    selected = rows[:n]
    summary = summaries[name]

    result = {
        "common_steps": n,
        "global_mpjpe_mm": mean(
            float(r["global_mpjpe_mm"]) for r in selected
        ),
        "root_relative_mpjpe_mm": mean(
            float(r["root_relative_mpjpe_mm"]) for r in selected
        ),
        "reset_count_in_100_steps": summary["reset_count"],
        "first_reset_time_s": summary["first_reset_time_s"],
        "reset_reasons": summary["reset_reason_counts"],
    }
    results[name] = result
    print(name, json.dumps(result, ensure_ascii=False))

(folder / "common_horizon_summary.json").write_text(
    json.dumps(results, indent=2)
)

print("SO SÁNH HAI CHECKPOINT HOÀN TẤT")
PY

↪**Phép so sánh chạy thành công**. Checkpoint 1000 tốt hơn checkpoint 500 trong điều kiện hiện tại, nhưng cả hai vẫn mất ổn định rất sớm.

| Chỉ số | `model_500.pt` | `model_1000.pt` |
|---|---:|---:|
| Global MPJPE trên cùng 7 bước đầu | 107,00 mm | **68,79 mm** |
| Root-relative MPJPE trên cùng 7 bước đầu | 105,02 mm | **62,43 mm** |
| Thời gian tới reset đầu tiên | 0,14 giây | **0,20 giây** |
| Số reset trong 100 bước | 12 | 10 |
| Nguyên nhân | `gravity_x` | `gravity_x` |

In [ ]:
from pathlib import Path
import zipfile
from google.colab import files

folder = Path("/content/hr17_checkpoint_comparison")
archive = Path("/content/HR17_ket_qua_so_sanh_checkpoint.zip")

assert (folder / "common_horizon_summary.json").exists()

with zipfile.ZipFile(
    archive, "w", compression=zipfile.ZIP_DEFLATED
) as package:
    for path in folder.rglob("*"):
        if path.is_file():
            package.write(path, arcname=path.relative_to("/content"))

files.download(str(archive))

In [ ]:
#Huấn luyện ngắn để kiểm tra quy trình
%%bash
set -euo pipefail

export PATH="/content/hr17_env/bin:$PATH"
export LD_LIBRARY_PATH="/content/hr17_env/lib:${LD_LIBRARY_PATH:-}"
export MAX_JOBS=2
export MPLBACKEND=Agg
export PYTHONHASHSEED=1

HR17_TRAIN_SCRIPT="/content/hr17_train_fresh.py"
if [ ! -f "$HR17_TRAIN_SCRIPT" ]; then
    HR17_TRAIN_SCRIPT="/hr17_train_fresh.py"
fi

test -f "$HR17_TRAIN_SCRIPT"
test -x /content/hr17_env/bin/python
test -f /content/hr17_checkpoint_comparison_input/env_cfg.json

cd /content/hr17_repo

/content/hr17_env/bin/python -u "$HR17_TRAIN_SCRIPT" \
  --seed 1 \
  --num-envs 128 \
  --iterations 200 \
  2>&1 | tee /content/hr17_training_console.log

| Chỉ số | Trung bình 20 vòng đầu | Trung bình 20 vòng cuối |
|---|---:|---:|
| Reward | −314,99 | −143,00 |
| Độ dài episode | 42,38 bước | 13,37 bước |

In [ ]:
from pathlib import Path
import json
import zipfile
from google.colab import files

pointer = Path("/content/hr17_training/latest_run.txt")
assert pointer.exists(), "Chưa tìm thấy thư mục lần huấn luyện."

run_dir = Path(pointer.read_text().strip())
metadata = json.loads((run_dir / "run_metadata.json").read_text())

assert metadata["status"] == "completed", (
    f"Lần chạy chưa hoàn tất: {metadata['status']}"
)

archive = Path("/content") / f"{run_dir.name}.zip"

with zipfile.ZipFile(
    archive, "w", compression=zipfile.ZIP_DEFLATED
) as package:
    for path in run_dir.rglob("*"):
        if path.is_file():
            package.write(
                path,
                arcname=Path(run_dir.name) / path.relative_to(run_dir),
            )

    console = Path("/content/hr17_training_console.log")
    if console.exists():
        package.write(
            console,
            arcname=Path(run_dir.name) / console.name,
        )

print("Thời gian chạy:", round(metadata["elapsed_wall_seconds"], 1), "giây")
files.download(str(archive))

In [ ]:
#So sánh policy trước và sau 200 vòng huấn luyện.

%%bash
set -euo pipefail

export PATH="/content/hr17_env/bin:$PATH"
export LD_LIBRARY_PATH="/content/hr17_env/lib:${LD_LIBRARY_PATH:-}"
export MAX_JOBS=2
export MPLBACKEND=Agg
export PYTHONHASHSEED=1

cd /content/hr17_repo

/content/hr17_env/bin/python -u /content/hr17_evaluate_fresh.py \
  --checkpoint model_initial.pt

/content/hr17_env/bin/python -u /content/hr17_evaluate_fresh.py \
  --checkpoint model_200.pt

| Chỉ số | Trước huấn luyện | Sau 200 vòng |
|---|---:|---:|
| Số bước đã chạy | 100 | 100 |
| Số lần reset | 1 | 7 |
| Reset đầu tiên | Bước 67 | Bước 13 |
| Thời gian tới reset đầu tiên | **1,34 giây** | **0,26 giây** |
| Nguyên nhân reset | `gravity_x` | `gravity_x` |

`gravity_x` nghĩa là thành phần trọng lực chiếu lên trục X của thân robot vượt ngưỡng kết thúc episode. Đây là dấu hiệu robot nghiêng quá mức theo điều kiện của môi trường.

In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()

expected_name = "hr17_evaluate_fresh.py"
assert expected_name in uploaded, (
    f"Bạn cần chọn đúng file {expected_name}"
)

destination = Path("/content") / expected_name
destination.write_bytes(uploaded[expected_name])

print("Đã lưu:", destination)
print("Dung lượng:", destination.stat().st_size, "bytes")

In [ ]:
#So sánh cùng 13 bước đầu

from pathlib import Path
import csv
import json
import statistics
import shutil
from google.colab import files

base = Path(
    "/content/hr17_fresh_evaluation/"
    "H2O_seed1_20261116_005911"
)

names = ["model_initial", "model_200"]
summaries = {}
frames = {}

for name in names:
    folder = base / name
    summaries[name] = json.loads(
        (folder / "summary.json").read_text()
    )
    with (folder / "frames.csv").open(newline="") as f:
        frames[name] = list(csv.DictReader(f))

# So sánh cùng đoạn đầu, gồm frame kết thúc
# episode của policy reset sớm hơn.
horizon = min(
    summaries[name]["first_reset_step"]
    or summaries[name]["frame_count"]
    for name in names
)

comparison = {
    "common_first_episode_steps": horizon,
    "includes_earliest_terminal_frame": True,
    "results": {},
}

for name in names:
    rows = frames[name][:horizon]
    assert len(rows) == horizon
    assert all(int(row["episode"]) == 0 for row in rows)

    result = {
        "global_mpjpe_mm": statistics.mean(
            float(r["global_mpjpe_mm"]) for r in rows
        ),
        "root_relative_mpjpe_mm": statistics.mean(
            float(r["root_relative_mpjpe_mm"]) for r in rows
        ),
        "first_reset_step":
            summaries[name]["first_reset_step"],
        "reset_count_100_steps":
            summaries[name]["reset_count"],
    }
    comparison["results"][name] = result

print(json.dumps(comparison, indent=2))

(base / "common_horizon_comparison.json").write_text(
    json.dumps(comparison, indent=2),
    encoding="utf-8",
)

zip_path = shutil.make_archive(
    "/content/HR17_danh_gia_truoc_sau_huan_luyen",
    "zip",
    root_dir=base.parent,
    base_dir=base.name,
)
files.download(zip_path)

**Kết quả xác nhận policy sau 200 vòng huấn luyện hoạt động kém hơn trong phép thử này**:

| Chỉ số | Trước huấn luyện | Sau 200 vòng |
|---|---:|---:|
| Global MPJPE trên cùng 13 bước đầu | **69,29 mm** | **98,82 mm** |
| Root-relative MPJPE trên cùng 13 bước đầu | **54,87 mm** | **87,38 mm** |
| Thời gian tới reset đầu tiên | **1,34 giây** | **0,26 giây** |
| Số reset trong 100 bước | **1** | **7** |


In [ ]:
#Đánh giá checkpoint 50, 100 và 150
from pathlib import Path
import re

required = [
    "hr17_evaluate_intermediate.py",
    "hr17_collect_checkpoint_timeline.py",
]

for name in required:
    matches = [
        key for key in uploaded
        if re.sub(r" \(\d+\)(?=\.py$)", "", Path(key).name) == name
    ]
    assert len(matches) == 1, f"Không xác định được file: {name}"

    destination = Path("/content") / name
    destination.write_bytes(uploaded[matches[0]])
    print("Đã lưu đúng tên:", destination)

In [ ]:
#Đánh giá ba checkpoint

%%bash
set -euo pipefail

export PATH="/content/hr17_env/bin:$PATH"
export LD_LIBRARY_PATH="/content/hr17_env/lib:${LD_LIBRARY_PATH:-}"
export MAX_JOBS=2
export MPLBACKEND=Agg
export PYTHONHASHSEED=1

cd /content/hr17_repo

for checkpoint in model_50.pt model_100.pt model_150.pt
do
  /content/hr17_env/bin/python -u \
    /content/hr17_evaluate_intermediate.py \
    --checkpoint "$checkpoint"
done

In [ ]:
import runpy

runpy.run_path(
    "/content/hr17_collect_checkpoint_timeline.py",
    run_name="__main__",
) #Kết hợp ba kết quả mới với kết quả trước huấn luyện và sau 200 vòng.
#Kiểm tra cấu hình đánh giá và mã SHA256 checkpoint.
#So sánh cả năm policy trên cùng đoạn đầu trước khi policy đầu tiên reset.

| Checkpoint | Thời gian tới reset đầu tiên | Global MPJPE trên cùng 12 bước đầu |
|---|---:|---:|
| Khởi tạo | 1,34 giây | **66,73 mm** |
| `model_50` | **1,50 giây** | 67,36 mm |
| `model_100` | 1,40 giây | 70,04 mm |
| `model_150` | 0,82 giây | 72,77 mm |
| `model_200` | 0,26 giây | 87,15 mm |

- `model_50` duy trì episode đầu lâu nhất trong các checkpoint đã kiểm tra. Các checkpoint sau giảm dần về chỉ số này, rõ nhất ở - `model_150` và `model_200`. Tuy nhiên, tất cả vẫn reset trước khi hoàn thành motion dài `5,233` giây.

In [ ]:
from google.colab import files
from pathlib import Path
import re

uploaded = files.upload()

matches = [
    name for name in uploaded
    if re.fullmatch(
        r"hr17_train_penalty025(?: \(\d+\))?\.py",
        Path(name).name,
    )
]
assert len(matches) == 1, "Hãy chọn đúng file hr17_train_penalty025.py"

destination = Path("/content/hr17_train_penalty025.py")
destination.write_bytes(uploaded[matches[0]])
print("Đã lưu:", destination)

In [ ]:
#Chạy lần huấn luyện đối chứng

%%bash
set -euo pipefail

export PATH="/content/hr17_env/bin:$PATH"
export LD_LIBRARY_PATH="/content/hr17_env/lib:${LD_LIBRARY_PATH:-}"
export MAX_JOBS=2
export MPLBACKEND=Agg
export PYTHONHASHSEED=1

cd /content/hr17_repo

/content/hr17_env/bin/python -u /content/hr17_train_penalty025.py \
  --seed 1 \
  --num-envs 128 \
  --iterations 200 \
  2>&1 | tee /content/hr17_penalty025_console.log

| Độ dài episode trung bình | Lần gốc: 1,0 | Lần mới: 0,25 |
|---|---:|---:|
| Trung bình 20 vòng cuối | 13,37 bước | **15,90 bước** |
| Quy đổi thời gian | 0,267 giây | **0,318 giây** |

- 200 cập nhật, 128 môi trường, 614.400 mẫu chuyển trạng thái.

- Thời gian khoảng 5 phút 8 giây.

- Đủ checkpoint và log; các scalar đọc được đều hữu hạn.

- Đối chiếu trực tiếp xác nhận chỉ đổi `penalty_scale` từ 1,0 xuống 0,25. Checkpoint khởi tạo giống từng byte; môi trường phần mềm và các bản vá giống lần trước.

↪ Có **tăng nhẹ** về độ dài episode trong huấn luyện, nhưng chưa thể kết luận policy ổn định hơn. Cả hai lần vẫn có xu hướng episode ngắn lại về cuối. Reward tổng cũng không dùng để so sánh trực tiếp vì ta đã đổi trọng số hàm thưởng.

In [ ]:
  from pathlib import Path
import json
import shutil
from google.colab import files

run_dir = Path(
    Path("/content/hr17_training/latest_penalty025_run.txt")
    .read_text().strip()
)

metadata = json.loads(
    (run_dir / "run_metadata.json").read_text()
)
assert metadata["status"] == "completed", (
    "Lần huấn luyện chưa hoàn thành"
)

console = Path("/content/hr17_penalty025_console.log")
if console.exists():
    shutil.copy2(console, run_dir / console.name)

zip_path = shutil.make_archive(
    str(Path("/content") / run_dir.name),
    "zip",
    root_dir=run_dir.parent,
    base_dir=run_dir.name,
)
files.download(zip_path)

In [ ]:
from pathlib import Path

paths = [
    "/content/hr17_env/bin/python",
    "/content/hr17_repo",
    "/content/hr17_training/H2O_penalty025_seed1_20261116_014337/model_200.pt",
]

for path in paths:
    print("CÒN" if Path(path).exists() else "THIẾU", path)

In [ ]:
#Khôi phục 2 lần huấn luyện và kết quả từ ZIP
from google.colab import files
from pathlib import Path
import re

names = [
    'H2O_seed1_20261116_005911.zip',
    'H2O_penalty025_seed1_20261116_014337.zip',
    'HR17_danh_gia_5_checkpoint.zip',
]
for name in names:
    destination = Path('/content') / name
    if destination.exists():
        print('Đã có ZIP, dùng file:', destination)
        continue
    print('Hãy chọn:', name)
    uploaded = files.upload()
    matches = [key for key in uploaded
               if re.sub(r' \(\d+\)(?=\.[^.]+$)', '', Path(key).name) == name]
    assert len(matches) == 1, 'Bạn cần chọn đúng file: ' + name
    destination.write_bytes(uploaded[matches[0]])

from pathlib import Path
import zipfile
import json
import hashlib
import shutil

def unpack(archive, destination):
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as package:
        assert package.testzip() is None, 'ZIP bị hỏng'
        for member in package.infolist():
            target = (destination / member.filename).resolve()
            assert target.is_relative_to(destination.resolve()), 'Đường dẫn ZIP không hợp lệ'
        package.extractall(destination)

training = Path('/content/hr17_training')
runs = ['H2O_seed1_20261116_005911', 'H2O_penalty025_seed1_20261116_014337']
for run in runs:
    unpack(Path('/content') / (run + '.zip'), training)
    folder = training / run
    meta = json.loads((folder / 'run_metadata.json').read_text())
    assert meta['status'] == 'completed'
    assert hashlib.sha256((folder / 'model_200.pt').read_bytes()).hexdigest() == meta['final_checkpoint_sha256']
    for script in folder.glob('hr17_train*.py'):
        shutil.copy2(script, Path('/content') / script.name)
    print('Đã khôi phục:', folder)

evaluation = Path('/content/hr17_fresh_evaluation')
unpack(Path('/content/HR17_danh_gia_5_checkpoint.zip'), evaluation)
for name in ['hr17_evaluate_fresh.py', 'hr17_evaluate_intermediate.py']:
    candidates = list(evaluation.rglob(name))
    assert candidates, 'Thiếu script: ' + name
    assert all(p.read_bytes() == candidates[0].read_bytes() for p in candidates)
    shutil.copy2(candidates[0], Path('/content') / name)

config_folder = Path('/content/hr17_checkpoint_comparison_input')
config_folder.mkdir(exist_ok=True)
baseline = training / runs[0]
for name in ['env_cfg.json', 'train_cfg.json']:
    shutil.copy2(baseline / ('source_' + name), config_folder / name)
(training / 'latest_run.txt').write_text(str(baseline))
(training / 'latest_penalty025_run.txt').write_text(str(training / runs[1]))
print('Đã khôi phục hai lần huấn luyện và kết quả năm checkpoint; không huấn luyện lại.')


In [ ]:
from google.colab import files
from pathlib import Path
import re

uploaded = files.upload()

matches = [
    name for name in uploaded
    if re.fullmatch(
        r"hr17_danh_gia_penalty025(?: \(\d+\))?\.py",
        Path(name).name,
    )
]
assert len(matches) == 1, "Hãy chọn đúng file hr17_danh_gia_penalty025.py"

destination = Path("/content/hr17_danh_gia_penalty025.py")
destination.write_bytes(uploaded[matches[0]])
print("Đã lưu:", destination)

In [ ]:
#Chạy đánh giá và so sánh hai lần huấn luyện
%%bash
set -euo pipefail

export PATH="/content/hr17_env/bin:$PATH"
export LD_LIBRARY_PATH="/content/hr17_env/lib:${LD_LIBRARY_PATH:-}"
export MAX_JOBS=2
export MPLBACKEND=Agg
export PYTHONHASHSEED=1

cd /content/hr17_repo

/content/hr17_env/bin/python -u \
  /content/hr17_danh_gia_penalty025.py

- Kiểm tra **checkpoint** và các bản vá đã khôi phục.
- Đánh giá **checkpoint 50, 100, 150, 200** của cả hai lần huấn luyện.
- Đánh giá thêm policy khởi tạo làm mốc; hai lần có checkpoint khởi tạo giống nhau.
- Chạy **100 bước mỗi checkpoint**, ghi dữ liệu trước reset.
- So sánh thời điểm reset và MPJPE trên cùng đoạn đầu.

| Checkpoint | Reset đầu: lần gốc | Reset đầu: lần 0,25 | Global MPJPE: gốc → mới |
|---|---:|---:|---:|
| `model_50` | **1,50 giây** | 1,36 giây | 67,36 → 75,12 mm |
| `model_100` | **1,40 giây** | 1,34 giây | 70,04 → 80,94 mm |
| `model_150` | **0,82 giây** | 0,26 giây | 72,77 → 98,62 mm |
| `model_200` | 0,26 giây | **0,36 giây** | **87,15 → 64,01 mm** |

In [ ]:
#Tải kết quả so sánh penalty025

from pathlib import Path
from google.colab import files

pointer = Path(
    "/content/hr17_penalty_comparison/latest_archive.txt"
)
assert pointer.exists(), "Ô đánh giá chưa hoàn thành."

archive = Path(pointer.read_text().strip())
assert archive.is_file(), "Chưa tìm thấy ZIP kết quả."

files.download(str(archive))

In [ ]:
#Kiểm tra môi trường và checkpoint SIM2SIM

from pathlib import Path
import os
import json
import hashlib
import subprocess

python = Path("/content/hr17_env/bin/python")
repo = Path("/content/hr17_repo")
run = Path(
    "/content/hr17_training/H2O_seed1_20261116_005911"
)

required = [
    python,
    repo / "legged_gym/legged_gym/envs/base/legged_robot.py",
    Path("/content/isaacgym/python/isaacgym/__init__.py"),
    run / "model_50.pt",
    run / "model_200.pt",
    run / "env_cfg.json",
    run / "train_cfg.json",
    run / "run_metadata.json",
]

missing = [str(path) for path in required if not path.is_file()]

if missing:
    print("CHƯA SẴN SÀNG. Thiếu các file:")
    for path in missing:
        print("-", path)
    raise RuntimeError(
        "Cần khôi phục môi trường hoặc gói huấn luyện trước."
    )

metadata = json.loads(
    (run / "run_metadata.json").read_text()
)
assert metadata["status"] == "completed"

final_sha = hashlib.sha256(
    (run / "model_200.pt").read_bytes()
).hexdigest()

assert final_sha == metadata["final_checkpoint_sha256"], (
    "Checkpoint model_200 không khớp metadata của lần huấn luyện."
)

env = os.environ.copy()
env["PATH"] = "/content/hr17_env/bin:" + env.get("PATH", "")
env["LD_LIBRARY_PATH"] = (
    "/content/hr17_env/lib:"
    "/content/isaacgym/python/isaacgym/_bindings/linux-x86_64:"
    + env.get("LD_LIBRARY_PATH", "")
)

check = subprocess.run(
    [
        str(python), "-u", "-c",
        "import isaacgym; import torch; "
        "print('PyTorch:', torch.__version__); "
        "print('CUDA:', torch.cuda.is_available()); "
        "assert torch.cuda.is_available(), 'Chua nhan GPU'; "
        "print('GPU:', torch.cuda.get_device_name(0))"
    ],
    cwd=str(repo),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print(check.stdout)
assert check.returncode == 0, "Kiểm tra môi trường chưa thành công."

checkpoint = run / "model_50.pt"
print("Checkpoint dự kiến:", checkpoint)
print(
    "SHA256:",
    hashlib.sha256(checkpoint.read_bytes()).hexdigest()
)
print("\nSẴN SÀNG CHO BƯỚC SIM2SIM")

In [ ]:
#SIM2SIM 2 — Tải mã thử nghiệm lên Colab
from google.colab import files
from pathlib import Path
import re

uploaded = files.upload()

matches = [
    name for name in uploaded
    if re.fullmatch(
        r"hr17_sim2sim_mass(?: \(\d+\))?\.py",
        Path(name).name
    )
]

assert len(matches) == 1, (
    "Hãy chọn đúng file hr17_sim2sim_mass.py"
)

destination = Path("/content/hr17_sim2sim_mass.py")
destination.write_bytes(uploaded[matches[0]])

print("ĐÃ TẢI MÃ:", destination)

In [ ]:
#SIM2SIM 3 — Chạy ba mức khối lượng
%%bash
set -euo pipefail

export PATH="/content/hr17_env/bin:$PATH"
export LD_LIBRARY_PATH="/content/hr17_env/lib:${LD_LIBRARY_PATH:-}"
export MAX_JOBS=2
export MPLBACKEND=Agg

/content/hr17_env/bin/python -u \
  /content/hr17_sim2sim_mass.py

In [ ]:
from pathlib import Path
import json

root = Path("/content/hr17_sim2sim_mass")
folders = sorted(
    [p for p in root.iterdir() if p.is_dir()],
    key=lambda p: p.stat().st_mtime,
) if root.exists() else []

if not folders:
    print("Chưa tạo được thư mục kết quả.")
    print(
        "Hãy kéo lên đầu output ô SIM2SIM 3 "
        "và gửi phần Traceback xuất hiện đầu tiên."
    )
else:
    latest = folders[-1]
    print("THƯ MỤC LẦN CHẠY:", latest)

    status = latest / "experiment_status.json"
    if status.is_file():
        print("\nTRẠNG THÁI:")
        print(status.read_text())

    for log in sorted(latest.glob("*.log")):
        print("\n" + "=" * 60)
        print("LOG:", log.name)
        print("=" * 60)
        lines = log.read_text(errors="replace").splitlines()
        print("\n".join(lines[-60:]))

In [ ]:
from pathlib import Path

root = Path("/content/hr17_sim2sim_mass")
folders = sorted(
    [p for p in root.iterdir() if p.is_dir()],
    key=lambda p: p.stat().st_mtime,
)

assert folders, "Chưa tìm thấy thư mục lần chạy."
latest = folders[-1]

print("LẦN CHẠY MỚI NHẤT:", latest.name)

status = latest / "experiment_status.json"
if status.is_file():
    print("\n=== TRẠNG THÁI VÀ LỖI ===")
    print(status.read_text())

for log in sorted(latest.glob("*.log")):
    print("\n=== LOG:", log.name, "===")
    lines = log.read_text(errors="replace").splitlines()
    print("\n".join(lines[-50:]))

In [ ]:
from pathlib import Path
from google.colab import files

folder = Path(
    "/content/hr17_sim2sim_mass/20261118_200451_615134"
)
archive = folder.with_suffix(".zip")

assert archive.is_file(), f"Không tìm thấy: {archive}"

print("Tải dữ liệu kiểm tra:", archive.name)
files.download(str(archive))

In [ ]:
# MA SÁT 1
from google.colab import files
from pathlib import Path
import re

uploaded = files.upload()

matches = [
    name for name in uploaded
    if re.fullmatch(
        r"hr17_sim2sim_friction(?: \(\d+\))?\.py",
        Path(name).name
    )
]

assert len(matches) == 1, (
    "Hãy chọn đúng file hr17_sim2sim_friction.py"
)

destination = Path("/content/hr17_sim2sim_friction.py")
destination.write_bytes(uploaded[matches[0]])

print("ĐÃ TẢI:", destination)



In [ ]:
#MA SÁT 2 — Chạy ba mức ma sát
from pathlib import Path
import os
import subprocess

python = Path("/content/hr17_env/bin/python")
script = Path("/content/hr17_sim2sim_friction.py")

assert python.is_file(), "Chưa khôi phục môi trường Python."
assert script.is_file(), "Chưa tải file ở ô MA SÁT 1."

env = os.environ.copy()
env["PATH"] = "/content/hr17_env/bin:" + env.get("PATH", "")
env["LD_LIBRARY_PATH"] = (
    "/content/hr17_env/lib:"
    "/content/isaacgym/python/isaacgym/_bindings/linux-x86_64:"
    + env.get("LD_LIBRARY_PATH", "")
)

log_path = Path("/content/hr17_friction_console.log")

with log_path.open("w", encoding="utf-8") as log:
    process = subprocess.Popen(
        [str(python), "-u", str(script)],
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)

    return_code = process.wait()

print("\nMã kết thúc:", return_code)
print("Log đầy đủ:", log_path)

if return_code != 0:
    print("Có lỗi. Hãy gửi phần cuối output để kiểm tra.")

**So sánh trên 72 bước chung trước reset, khoảng 1,44 giây:**

| Ma sát robot và nền | Global MPJPE ↓ | Thời gian đến reset ↑ |
|---|---:|---:|
| 0,5 | 167,63 mm | 1,46 giây |
| 1,0 — gốc | 154,60 mm | 1,50 giây |
| 1,5 | 152,65 mm | 1,50 giây |

↪**Giảm ma sát xuống 0,5** làm **sai số tăng 8,42%** và robot reset sớm hơn. **Tăng lên 1,5 giảm sai số nhẹ 1,26%**, nhưng không kéo dài thời gian đến reset.

Cả ba trường hợp vẫn reset do nghiêng thân, chưa hoàn thành motion. Kết quả mới phản ánh một motion và một seed.

In [ ]:
from pathlib import Path
from google.colab import files

pointer = Path(
    "/content/hr17_sim2sim_friction/latest_archive.txt"
)
assert pointer.is_file(), "Chưa tìm thấy gói kết quả."

archive = Path(pointer.read_text().strip())
assert archive.is_file(), f"Không tìm thấy: {archive}"

print("Tải kết quả:", archive.name)
files.download(str(archive))

# **Kiểm tra độ trễ điều khiển**

| Điều kiện | Trễ thêm |
|---|---:|
| `delay0` | 0 ms — đối chứng |
| `delay1` | Khoảng 20 ms |
| `delay2` | Khoảng 40 ms |
| `delay3` | Khoảng 60 ms |

In [ ]:
#ĐỘ TRỄ 1 — Tải mã
from google.colab import files
from pathlib import Path
import re

uploaded = files.upload()

matches = [
    name for name in uploaded
    if re.fullmatch(
        r"hr17_sim2sim_delay(?: \(\d+\))?\.py",
        Path(name).name
    )
]

assert len(matches) == 1, (
    "Hãy chọn đúng file hr17_sim2sim_delay.py"
)

destination = Path("/content/hr17_sim2sim_delay.py")
destination.write_bytes(uploaded[matches[0]])

print("ĐÃ TẢI:", destination)

In [ ]:
#ĐỘ TRỄ 2 — Chạy bốn điều kiện
from pathlib import Path
import os
import subprocess

python = Path("/content/hr17_env/bin/python")
script = Path("/content/hr17_sim2sim_delay.py")

assert python.is_file(), "Chưa khôi phục môi trường Python."
assert script.is_file(), "Chưa tải mã ở ô ĐỘ TRỄ 1."

env = os.environ.copy()
env["PATH"] = "/content/hr17_env/bin:" + env.get("PATH", "")
env["LD_LIBRARY_PATH"] = (
    "/content/hr17_env/lib:"
    "/content/isaacgym/python/isaacgym/_bindings/linux-x86_64:"
    + env.get("LD_LIBRARY_PATH", "")
)

log_path = Path("/content/hr17_delay_console.log")

with log_path.open("w", encoding="utf-8") as log:
    process = subprocess.Popen(
        [str(python), "-u", str(script)],
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)

    return_code = process.wait()

print("\nMã kết thúc:", return_code)
print("Log đầy đủ:", log_path)

if return_code != 0:
    print("Có lỗi. Hãy gửi phần cuối output để kiểm tra.")

**So sánh trên 66 bước chung trước reset, khoảng 1,32 giây:**

| Trễ thêm | Global MPJPE ↓ | Thời gian đến reset ↑ |
|---|---:|---:|
| 0 ms | 126,94 mm | 1,50 giây |
| 20 ms | 113,43 mm | 1,58 giây |
| 40 ms | 138,55 mm | 1,44 giây |
| 60 ms | 174,01 mm | 1,34 giây |



In [ ]:
#ĐỘ TRỄ 3 — Tải kết quả
from pathlib import Path
from google.colab import files

pointer = Path(
    "/content/hr17_sim2sim_delay/latest_archive.txt"
)
assert pointer.is_file(), "Chưa tìm thấy gói kết quả."

archive = Path(pointer.read_text().strip())
assert archive.is_file(), f"Không tìm thấy: {archive}"

print("Tải kết quả:", archive.name)
files.download(str(archive))

In [ ]:
#HUẤN LUYỆN 3 SEED — Kiểm tra dữ liệu gốc
from pathlib import Path
import json
import hashlib

run = Path(
    "/content/hr17_training/H2O_seed1_20261116_005911"
)

required = [
    Path("/content/hr17_env/bin/python"),
    run / "run_metadata.json",
    run / "source_env_cfg.json",
    run / "source_train_cfg.json",
    run / "hr17_train_fresh.py",
    run / "model_200.pt",
]

missing = [str(p) for p in required if not p.is_file()]
assert not missing, "Cần khôi phục các file:\n" + "\n".join(missing)

metadata = json.loads(
    (run / "run_metadata.json").read_text()
)

assert metadata["status"] == "completed"
assert metadata["seed"] == 1
assert metadata["num_envs"] == 128
assert metadata["iterations"] == 200

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha256(run / "model_200.pt") == (
    metadata["final_checkpoint_sha256"]
), "Checkpoint seed 1 không khớp."

assert sha256(run / "hr17_train_fresh.py") == (
    metadata["training_script_sha256"]
), "Script huấn luyện không khớp bản đã dùng cho seed 1."

print("Seed 1: đã hoàn thành")
print("Số môi trường:", metadata["num_envs"])
print("Số cập nhật:", metadata["iterations"])
print("Checkpoint và script: khớp SHA256")
print("\nSẴN SÀNG CHUẨN BỊ SEED 2 VÀ SEED 3")

In [ ]:
#BA SEED 2 — Tải chương trình
from google.colab import files
from pathlib import Path
import re

uploaded = files.upload()

matches = [
    name for name in uploaded
    if re.fullmatch(
        r"hr17_train_three_seeds(?: \(\d+\))?\.py",
        Path(name).name
    )
]

assert len(matches) == 1, (
    "Hãy chọn đúng file hr17_train_three_seeds.py"
)

destination = Path("/content/hr17_train_three_seeds.py")
destination.write_bytes(uploaded[matches[0]])

print("ĐÃ TẢI:", destination)

In [ ]:
#BA SEED 3 — Huấn luyện seed 2 rồi seed 3
from pathlib import Path
import os
import subprocess

python = Path("/content/hr17_env/bin/python")
script = Path("/content/hr17_train_three_seeds.py")

assert python.is_file(), "Chưa khôi phục môi trường."
assert script.is_file(), "Chưa tải chương trình."

env = os.environ.copy()
env["PATH"] = "/content/hr17_env/bin:" + env.get("PATH", "")
env["LD_LIBRARY_PATH"] = (
    "/content/hr17_env/lib:"
    "/content/isaacgym/python/isaacgym/_bindings/linux-x86_64:"
    + env.get("LD_LIBRARY_PATH", "")
)

log_path = Path("/content/hr17_three_seeds_console.log")

with log_path.open("w", encoding="utf-8") as log:
    process = subprocess.Popen(
        [
            str(python), "-u", str(script),
            "--seeds", "2", "3"
        ],
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)

    return_code = process.wait()

print("\nMã kết thúc:", return_code)
print("Log đầy đủ:", log_path)

if return_code != 0:
    print("Có lỗi. Hãy gửi phần cuối output để kiểm tra.")

In [ ]:
  #BA SEED 4 — Tải kết quả
  from pathlib import Path
from google.colab import files

pointer = Path(
    "/content/hr17_three_seeds/latest_archive.txt"
)
assert pointer.is_file(), "Chưa tìm thấy gói kết quả."

archive = Path(pointer.read_text().strip())
assert archive.is_file(), f"Không tìm thấy: {archive}"

print("Tải kết quả:", archive.name)
files.download(str(archive))

 **Thống kê trung bình 20 cập nhật cuối:**

| Seed | Thời gian huấn luyện | Mean reward | Độ dài episode |
|---|---:|---:|---:|
| 1 | 313,91 giây | −143,00 | 13,368 bước |
| 2 | 328,83 giây | −83,79 | 9,031 bước |
| 3 | 336,98 giây | −92,66 | 10,772 bước |

In [ ]:
#1 — Tải script đánh giá lên Colab
from google.colab import files
from pathlib import Path
import re

uploaded = files.upload()

matches = [
    name for name in uploaded
    if re.fullmatch(
        r"hr17_evaluate_three_seeds(?: \(\d+\))?\.py",
        name
    )
]
assert len(matches) == 1, "Hãy chọn đúng file hr17_evaluate_three_seeds.py"

destination = Path("/content/hr17_evaluate_three_seeds.py")
destination.write_bytes(uploaded[matches[0]])

print("ĐÃ LƯU SCRIPT:", destination)

In [ ]:
#2 — Đánh giá sáu checkpoint
from pathlib import Path
import os
import subprocess

script = Path("/content/hr17_evaluate_three_seeds.py")
python = Path("/content/hr17_env/bin/python")

assert script.is_file(), "Hãy chạy ô tải script trước."
assert python.is_file(), "Cần khôi phục môi trường Colab trước."

env = os.environ.copy()
env["PATH"] = "/content/hr17_env/bin:" + env.get("PATH", "")
env["LD_LIBRARY_PATH"] = (
    "/content/hr17_env/lib:"
    "/content/isaacgym/python/isaacgym/_bindings/linux-x86_64:"
    + env.get("LD_LIBRARY_PATH", "")
)
env["MAX_JOBS"] = "2"
env["MPLBACKEND"] = "Agg"
env["PYTHONHASHSEED"] = "1"
env["PYTHONUNBUFFERED"] = "1"

log_path = Path("/content/hr17_three_seed_evaluation_console.log")

with log_path.open("w", encoding="utf-8") as log:
    process = subprocess.Popen(
        [str(python), "-u", str(script)],
        cwd="/content/hr17_repo",
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)

    return_code = process.wait()

print("\nMã kết thúc:", return_code)
print("Log:", log_path)

**Kết quả sau huấn luyện hiện kém hơn bản khởi tạo:**

| Seed | Global MPJPE trước → sau ↓ | Thời gian đến reset trước → sau ↑ |
|---|---:|---:|
| 1 | 45,18 → 52,76 mm | 1,34 → 0,26 giây |
| 2 | 45,66 → 98,44 mm | 1,72 → 0,12 giây |
| 3 | 42,49 → 55,74 mm | 1,24 → 0,20 giây |

In [ ]:
#3 — Tải kết quả về máy
from google.colab import files
from pathlib import Path

pointer = Path("/content/hr17_three_seed_evaluation/latest_archive.txt")
assert pointer.is_file(), "Chưa tìm thấy gói kết quả; hãy xem output ô 2."

archive = Path(pointer.read_text().strip())
assert archive.is_file(), f"Không tìm thấy: {archive}"

print("Tải gói kết quả:", archive.name)
files.download(str(archive))

In [ ]:
#tải script chẩn đoán
from google.colab import files
from pathlib import Path
import re

uploaded = files.upload()

matches = [
    name for name in uploaded
    if re.fullmatch(
        r"hr17_diagnose_rewards(?: \(\d+\))?\.py",
        name
    )
]

assert len(matches) == 1, "Hãy chọn file hr17_diagnose_rewards.py"

destination = Path("/content/hr17_diagnose_rewards.py")
destination.write_bytes(uploaded[matches[0]])

print("ĐÃ LƯU:", destination)

In [ ]:
#chạy chẩn đoán reward
from pathlib import Path
import os
import subprocess

script = Path("/content/hr17_diagnose_rewards.py")
python = Path("/content/hr17_env/bin/python")

assert script.is_file(), "Hãy tải script ở ô 1 trước."
assert python.is_file(), "Cần khôi phục môi trường trước."

env = os.environ.copy()
env["PATH"] = "/content/hr17_env/bin:" + env.get("PATH", "")
env["LD_LIBRARY_PATH"] = (
    "/content/hr17_env/lib:"
    "/content/isaacgym/python/isaacgym/_bindings/linux-x86_64:"
    + env.get("LD_LIBRARY_PATH", "")
)
env["MAX_JOBS"] = "2"
env["MPLBACKEND"] = "Agg"
env["PYTHONHASHSEED"] = "1"
env["PYTHONUNBUFFERED"] = "1"

log_path = Path("/content/hr17_reward_diagnosis_console.log")

with log_path.open("w", encoding="utf-8") as log:
    process = subprocess.Popen(
        [str(python), "-u", str(script)],
        cwd="/content/hr17_repo",
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)

    return_code = process.wait()

print("\nMã kết thúc:", return_code)
print("Log:", log_path)

- Đã đối chiếu 95 tệp trong danh mục kiểm tra.

- Checkpoint, trạng thái ban đầu và tọa độ chuyển động khớp với lần đánh giá trước.

- Tổng các thành phần reward khớp tổng reward từng bước, sai số tối đa khoảng 0,0000035.

**Reward trung bình trên cùng 5 bước đầu, khoảng 0,10 giây:**

| Seed | Trước huấn luyện | Sau 200 cập nhật |
|---|---:|---:|
| 1 | −4,2733 | −3,4889 |
| 2 | −4,4791 | −8,9783 |
| 3 | −3,8615 | −3,4905 |

In [ ]:
#tải kết quả về máy
from google.colab import files
from pathlib import Path

pointer = Path("/content/hr17_reward_diagnosis/latest_archive.txt")
assert pointer.is_file(), "Chưa có gói kết quả; hãy xem output ô 2."

archive = Path(pointer.read_text().strip())
assert archive.is_file(), f"Không tìm thấy: {archive}"

print("Tải gói:", archive.name)
files.download(str(archive))